# Fabric Defect Classification — 03. Hyperparameter Tuning

This notebook performs a **structured grid search** over the Improved CNN from `02_Improved_CNN.ipynb`. Each configuration is trained for 12 epochs on the training split and scored on the held-out validation split; the best configuration is then retrained from scratch for 25 epochs and evaluated on the test split.

## Search space

| Hyperparameter | Values searched | Why it matters |
|---|---|---|
| Learning rate | 1e-3, 5e-4, 1e-4 | Too high → training diverges; too low → underfits in the epoch budget |
| Batch size | 32, 64 | Larger batches give smoother gradients but worse generalisation on small datasets |
| Dropout rate | 0.3, 0.4, 0.5 | Counterweight to BatchNorm's implicit regularisation |
| Optimiser | Adam, SGD-momentum | SGD often generalises better; Adam converges faster |
| Weight decay | 1e-4, 5e-4 | L2 regularisation strength |
| LR schedule | CosineAnnealingLR, StepLR, CosineAnnealingWarmRestarts | Different annealing schedules suit different optimisers |

Total = `3 × 2 × 3 × 2 × 2 × 3 = 216` configurations. Each grid-search run is short (12 epochs) and results are streamed to `grid_search_results.json` so the search resumes after interruption.

## Data-leakage controls
* **The split is fixed.** All 216 runs use the identical `fabric_split/{train, val, test}/` data prepared in notebook 00. We do not re-split anywhere.
* **Test set untouched during search.** Every grid-search run trains on `train`, validates on `val`. The `test` set is read exactly once at the end, after the winning config has been chosen and retrained.
* **Selection criterion = validation accuracy.** Picking the configuration on test accuracy would constitute test-set leakage — the test set must be a single-use ruler.
* Class weights are computed once, from training labels only, and reused across all runs.

## 1. Imports & setup

In [1]:
import os
import copy
import json
import time
import random
import itertools
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from sklearn.metrics import (
    confusion_matrix, classification_report, precision_recall_fscore_support, roc_curve, auc
)

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else
                      'mps'  if torch.backends.mps.is_available() else 'cpu')
print(f'Using device: {device}')

sns.set_style('whitegrid')
plt.rcParams['figure.dpi'] = 110

Using device: cuda


## 2. Dataset — same split as the other notebooks

In [2]:
DATA_DIR = 'fabric_split'
IMG_SIZE = 128
MEAN = [0.485, 0.456, 0.406]
STD  = [0.229, 0.224, 0.225]

data_transforms = {
    'train': transforms.Compose([
        transforms.Resize((IMG_SIZE, IMG_SIZE)),
        transforms.RandomHorizontalFlip(),
        transforms.RandomRotation(15),
        transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.1),
        transforms.ToTensor(),
        transforms.Normalize(MEAN, STD),
    ]),
    'val':  transforms.Compose([transforms.Resize((IMG_SIZE, IMG_SIZE)),
                                 transforms.ToTensor(),
                                 transforms.Normalize(MEAN, STD)]),
    'test': transforms.Compose([transforms.Resize((IMG_SIZE, IMG_SIZE)),
                                 transforms.ToTensor(),
                                 transforms.Normalize(MEAN, STD)]),
}

image_datasets = {
    s: datasets.ImageFolder(os.path.join(DATA_DIR, s), data_transforms[s])
    for s in ['train', 'val', 'test']
}
dataset_sizes = {s: len(image_datasets[s]) for s in ['train', 'val', 'test']}
class_names   = image_datasets['train'].classes
num_classes   = len(class_names)

print('Classes      :', class_names)
print('Dataset sizes:', dataset_sizes)

Classes      : ['hole', 'horizontal', 'stain', 'vertical']
Dataset sizes: {'train': 640, 'val': 138, 'test': 138}


## 3. Class-imbalance weights (from training set only)

In [3]:
train_labels = [label for _, label in image_datasets['train'].samples]
class_counts = Counter(train_labels)
total        = sum(class_counts.values())
class_weights = [total / (num_classes * class_counts[i]) for i in range(num_classes)]
class_weights_tensor = torch.FloatTensor(class_weights).to(device)
print('Class weights:', dict(zip(class_names, [round(w, 4) for w in class_weights])))

Class weights: {'hole': 0.8122, 'horizontal': 1.7021, 'stain': 0.5755, 'vertical': 2.2535}


## 4. Tunable Improved CNN

Identical architecture to the Improved CNN, but the `dropout_rate` is exposed as a constructor argument so each grid-search run can vary it.

In [4]:
class TunableImprovedCNN(nn.Module):
    def __init__(self, num_classes: int = 4, dropout_rate: float = 0.4):
        super().__init__()
        def block(i, o):
            return nn.Sequential(
                nn.Conv2d(i, o, 3, padding=1, bias=False), nn.BatchNorm2d(o), nn.ReLU(inplace=True),
                nn.Conv2d(o, o, 3, padding=1, bias=False), nn.BatchNorm2d(o), nn.ReLU(inplace=True),
                nn.MaxPool2d(2, 2),
            )
        self.features = nn.Sequential(
            block(3,    32),
            block(32,   64),
            block(64,  128),
            block(128, 256),
            nn.AdaptiveAvgPool2d(1),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Dropout(dropout_rate),
            nn.Linear(256, 128), nn.ReLU(inplace=True),
            nn.Dropout(dropout_rate),
            nn.Linear(128, num_classes),
        )
    def forward(self, x):
        return self.classifier(self.features(x))

# Sanity check
dummy = torch.zeros(2, 3, IMG_SIZE, IMG_SIZE)
test_model = TunableImprovedCNN(num_classes=num_classes)
print('Output shape:', test_model(dummy).shape)

Output shape: torch.Size([2, 4])


## 5. Helper functions

In [5]:
def build_optimizer(model, config: dict):
    lr, wd, name = config['lr'], config['weight_decay'], config['optimizer']
    if name == 'Adam':
        return optim.Adam(model.parameters(), lr=lr, weight_decay=wd)
    if name == 'SGD':
        return optim.SGD(model.parameters(), lr=lr, momentum=0.9, weight_decay=wd)
    raise ValueError(name)


def build_scheduler(optimizer, config: dict, num_epochs: int):
    name = config['lr_schedule']
    if name == 'CosineAnnealingLR':
        return optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=num_epochs)
    if name == 'StepLR':
        return optim.lr_scheduler.StepLR(optimizer, step_size=4, gamma=0.5)
    if name == 'CosineAnnealingWarmRestarts':
        return optim.lr_scheduler.CosineAnnealingWarmRestarts(
            optimizer, T_0=4, T_mult=1, eta_min=config['lr'] * 0.01
        )
    raise ValueError(name)


def train_one_config(config: dict, num_epochs: int, verbose: bool = False):
    """Train one hyperparameter config; return best val acc, full history, best weights."""
    batch_size   = config['batch_size']
    dropout_rate = config['dropout_rate']

    loaders = {
        split: DataLoader(image_datasets[split], batch_size=batch_size,
                          shuffle=(split == 'train'), num_workers=0)
        for split in ['train', 'val']
    }
    model     = TunableImprovedCNN(num_classes=num_classes, dropout_rate=dropout_rate).to(device)
    criterion = nn.CrossEntropyLoss(weight=class_weights_tensor)
    optimizer = build_optimizer(model, config)
    scheduler = build_scheduler(optimizer, config, num_epochs)

    best_acc = 0.0
    best_wts = copy.deepcopy(model.state_dict())
    history  = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': [], 'lr': []}

    for epoch in range(num_epochs):
        for phase in ['train', 'val']:
            model.train() if phase == 'train' else model.eval()
            running_loss, running_corrects = 0.0, 0
            for inputs, labels in loaders[phase]:
                inputs, labels = inputs.to(device), labels.to(device)
                optimizer.zero_grad()
                with torch.set_grad_enabled(phase == 'train'):
                    outputs = model(inputs)
                    _, preds = torch.max(outputs, 1)
                    loss = criterion(outputs, labels)
                    if phase == 'train':
                        loss.backward()
                        optimizer.step()
                running_loss     += loss.item() * inputs.size(0)
                running_corrects += torch.sum(preds == labels.data)
            epoch_loss = running_loss / dataset_sizes[phase]
            epoch_acc  = (running_corrects.float() / dataset_sizes[phase]).item()
            history[f'{phase}_loss'].append(epoch_loss)
            history[f'{phase}_acc'].append(epoch_acc)
            if phase == 'val' and epoch_acc > best_acc:
                best_acc = epoch_acc
                best_wts = copy.deepcopy(model.state_dict())
        history['lr'].append(optimizer.param_groups[0]['lr'])
        scheduler.step()
        if verbose:
            print(f'  epoch {epoch+1:02d}  val_loss={history["val_loss"][-1]:.4f}  val_acc={history["val_acc"][-1]:.4f}')
    return {'best_val_acc': best_acc, 'history': history, 'best_weights': best_wts}

## 6. Grid search

Each configuration is tagged with its full hyperparameter dict and its result is appended to a JSON log on disk. If the kernel dies, re-running this cell will resume from where it left off.

**Time estimate (CPU):** 216 configs × ~3–5 minutes each = a long-running job. If your machine has no GPU and you want a smaller run, edit `GRID` below to a coarser space.

In [6]:
GRID = {
    'lr':           [1e-3, 5e-4, 1e-4],
    'batch_size':   [32, 64],
    'dropout_rate': [0.3, 0.4, 0.5],
    'optimizer':    ['Adam', 'SGD'],
    'weight_decay': [1e-4, 5e-4],
    'lr_schedule':  ['CosineAnnealingLR', 'StepLR', 'CosineAnnealingWarmRestarts'],
}
GRID_SEARCH_EPOCHS = 12
RESULTS_FILE       = 'grid_search_results.json'

keys   = list(GRID.keys())
combos = list(itertools.product(*GRID.values()))
print(f'Total configurations: {len(combos)}')

# Resume support
if os.path.exists(RESULTS_FILE):
    with open(RESULTS_FILE) as f:
        all_results = json.load(f)
    print(f'Resuming — {len(all_results)} runs already on disk.')
else:
    all_results = []

Total configurations: 216


In [ ]:
from tqdm.auto import tqdm

completed = {json.dumps(r['config'], sort_keys=True) for r in all_results}

for idx, combo in enumerate(tqdm(combos, desc='Grid search')):
    config = dict(zip(keys, combo))
    key    = json.dumps(config, sort_keys=True)
    if key in completed:
        continue
    tqdm.write(f'\n[{idx+1}/{len(combos)}] {config}')
    t0 = time.time()
    res = train_one_config(config, num_epochs=GRID_SEARCH_EPOCHS, verbose=False)
    elapsed = time.time() - t0
    tqdm.write(f'   best val acc = {res["best_val_acc"]:.4f}   ({elapsed:.0f}s)')

    all_results.append({
        'config':       config,
        'best_val_acc': res['best_val_acc'],
        'history':      res['history'],
        'elapsed_s':    elapsed,
    })
    with open(RESULTS_FILE, 'w') as f:
        json.dump(all_results, f, indent=2)

print('\nGrid search complete.')

C:\Users\user\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Grid search:   0%|                                                                             | 0/216 [00:00<?, ?it/s]


[1/216] {'lr': 0.001, 'batch_size': 32, 'dropout_rate': 0.3, 'optimizer': 'Adam', 'weight_decay': 0.0001, 'lr_schedule': 'CosineAnnealingLR'}


Grid search:   0%|▎                                                                  | 1/216 [01:31<5:28:11, 91.59s/it]

   best val acc = 0.9058   (92s)

[2/216] {'lr': 0.001, 'batch_size': 32, 'dropout_rate': 0.3, 'optimizer': 'Adam', 'weight_decay': 0.0001, 'lr_schedule': 'StepLR'}


Grid search:   1%|▌                                                                  | 2/216 [03:02<5:24:58, 91.11s/it]

...

   best val acc = 0.8696   (112s)

[174/216] {'lr': 0.0001, 'batch_size': 32, 'dropout_rate': 0.5, 'optimizer': 'Adam', 'weight_decay': 0.0005, 'lr_schedule': 'CosineAnnealingWarmRestarts'}


Grid search:  81%|█████████████████████████████████████████████████▉            | 174/216 [4:23:45<1:18:11, 111.71s/it]

   best val acc = 0.8623   (109s)

[175/216] {'lr': 0.0001, 'batch_size': 32, 'dropout_rate': 0.5, 'optimizer': 'SGD', 'weight_decay': 0.0001, 'lr_schedule': 'CosineAnnealingLR'}


Grid search:  81%|██████████████████████████████████████████████████▏           | 175/216 [4:25:27<1:14:19, 108.78s/it]

   best val acc = 0.6594   (102s)

[176/216] {'lr': 0.0001, 'batch_size': 32, 'dropout_rate': 0.5, 'optimizer': 'SGD', 'weight_decay': 0.0001, 'lr_schedule': 'StepLR'}


## 7. Results analysis

### 7.1 Top configurations

In [ ]:
results_df = pd.DataFrame([
    {**r['config'], 'best_val_acc': r['best_val_acc'], 'elapsed_s': r.get('elapsed_s')}
    for r in all_results
]).sort_values('best_val_acc', ascending=False).reset_index(drop=True)

print('Top 10 configurations by validation accuracy:')
print(results_df.head(10).to_string(index=False))

best_row = results_df.iloc[0]
best_config = {
    'lr'          : float(best_row['lr']),
    'batch_size'  : int(best_row['batch_size']),
    'dropout_rate': float(best_row['dropout_rate']),
    'optimizer'   : str(best_row['optimizer']),
    'weight_decay': float(best_row['weight_decay']),
    'lr_schedule' : str(best_row['lr_schedule']),
}
print('\nBest configuration:')
for k, v in best_config.items():
    print(f'  {k:<14s}: {v}')

### 7.2 Hyperparameter sensitivity

For each hyperparameter, group all runs by its value and compute mean / max / std of best validation accuracy. The taller and more separated the bars, the more impact that hyperparameter has.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 9))
axes = axes.flatten()

for ax_i, param in enumerate(keys):
    grouped = results_df.groupby(param)['best_val_acc'].agg(['mean', 'max', 'std']).reset_index()
    grouped = grouped.sort_values('mean', ascending=False)
    x_labels = grouped[param].astype(str)
    x_pos = np.arange(len(x_labels))

    axes[ax_i].bar(x_pos, grouped['mean'], yerr=grouped['std'].fillna(0), capsize=5,
                   color='steelblue', alpha=0.75, label='Mean ± Std')
    axes[ax_i].scatter(x_pos, grouped['max'], color='darkorange', zorder=5, s=70, label='Max')
    axes[ax_i].set_xticks(x_pos)
    axes[ax_i].set_xticklabels(x_labels, rotation=15, ha='right', fontsize=9)
    axes[ax_i].set_title(param, fontweight='bold')
    axes[ax_i].set_ylabel('Best val accuracy')
    axes[ax_i].legend(fontsize=8); axes[ax_i].grid(axis='y', alpha=0.3)

plt.suptitle('Hyperparameter sensitivity — mean & max best val acc', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('hp_sensitivity.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Box plots — give a clearer picture of distribution shape than bar charts alone
fig, axes = plt.subplots(2, 3, figsize=(16, 9))
axes = axes.flatten()
for ax_i, param in enumerate(keys):
    sns.boxplot(x=param, y='best_val_acc', data=results_df, ax=axes[ax_i], palette='viridis', hue=param, legend=False)
    axes[ax_i].set_title(param, fontweight='bold')
    axes[ax_i].tick_params(axis='x', rotation=15)
    axes[ax_i].grid(axis='y', alpha=0.3)
plt.suptitle('Distribution of val accuracy across hyperparameter values', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('hp_boxplots.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Heatmap — interaction between two of the most impactful hyperparameters
pivot = results_df.pivot_table(index='lr', columns='optimizer', values='best_val_acc', aggfunc='mean')
fig, ax = plt.subplots(figsize=(7, 4.5))
sns.heatmap(pivot, annot=True, fmt='.4f', cmap='YlGnBu', ax=ax, cbar_kws={'label': 'Mean val acc'})
ax.set_title('Interaction: learning rate × optimiser', fontweight='bold')
plt.tight_layout()
plt.savefig('hp_lr_optimiser_interaction.png', dpi=150, bbox_inches='tight')
plt.show()

pivot2 = results_df.pivot_table(index='dropout_rate', columns='weight_decay', values='best_val_acc', aggfunc='mean')
fig, ax = plt.subplots(figsize=(7, 4.5))
sns.heatmap(pivot2, annot=True, fmt='.4f', cmap='YlGnBu', ax=ax, cbar_kws={'label': 'Mean val acc'})
ax.set_title('Interaction: dropout × weight decay', fontweight='bold')
plt.tight_layout()
plt.savefig('hp_dropout_wd_interaction.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Histogram of val accuracy across all 216 runs — are most runs good, or only a few?
fig, ax = plt.subplots(figsize=(9, 4.5))
ax.hist(results_df['best_val_acc'], bins=25, color='steelblue', edgecolor='white', alpha=0.85)
ax.axvline(results_df['best_val_acc'].max(), color='red', linestyle='--', linewidth=2,
           label=f'Best = {results_df["best_val_acc"].max():.4f}')
ax.axvline(results_df['best_val_acc'].mean(), color='green', linestyle='--', linewidth=2,
           label=f'Mean = {results_df["best_val_acc"].mean():.4f}')
ax.set_xlabel('Best val accuracy'); ax.set_ylabel('Number of configurations')
ax.set_title('Distribution of best val accuracy across grid-search runs', fontweight='bold')
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('hp_val_acc_histogram.png', dpi=150, bbox_inches='tight')
plt.show()

### 7.3 Learning curves of the best grid-search run

In [ ]:
best_key = json.dumps(best_config, sort_keys=True)
best_run = next(r for r in all_results if json.dumps(r['config'], sort_keys=True) == best_key)
hist = best_run['history']

fig, axes = plt.subplots(1, 3, figsize=(16, 4.2))
axes[0].plot(hist['train_loss'], '-o', label='Train'); axes[0].plot(hist['val_loss'], '-s', label='Val')
axes[0].set_title('Loss', fontweight='bold'); axes[0].set_xlabel('Epoch'); axes[0].legend(); axes[0].grid(alpha=0.3)
axes[1].plot(hist['train_acc'], '-o', label='Train'); axes[1].plot(hist['val_acc'], '-s', label='Val')
axes[1].set_title('Accuracy', fontweight='bold'); axes[1].set_xlabel('Epoch'); axes[1].legend(); axes[1].grid(alpha=0.3)
axes[2].plot(hist['lr'], '-o', color='purple')
axes[2].set_title('Learning rate', fontweight='bold'); axes[2].set_xlabel('Epoch'); axes[2].grid(alpha=0.3)
plt.suptitle(f'Best grid-search run — {GRID_SEARCH_EPOCHS} epochs', fontweight='bold')
plt.tight_layout()
plt.savefig('hp_best_grid_curves.png', dpi=150, bbox_inches='tight')
plt.show()

## 8. Full retraining of the best configuration

Now we retrain the winner from a fresh initialisation for the full 25 epochs and use that final model for test-set evaluation.

In [ ]:
FULL_EPOCHS = 25
print(f'Retraining best config for {FULL_EPOCHS} epochs:')
for k, v in best_config.items():
    print(f'  {k:<14s}: {v}')

full_result = train_one_config(best_config, num_epochs=FULL_EPOCHS, verbose=True)
print(f'\nFinal val acc after full retraining: {full_result["best_val_acc"]:.4f}')

torch.save(full_result['best_weights'], 'tuned_cnn_weights.pth')

In [ ]:
hist_full = full_result['history']
fig, axes = plt.subplots(1, 3, figsize=(16, 4.2))
axes[0].plot(hist_full['train_loss'], '-o', label='Train'); axes[0].plot(hist_full['val_loss'], '-s', label='Val')
axes[0].set_title('Loss', fontweight='bold'); axes[0].set_xlabel('Epoch'); axes[0].legend(); axes[0].grid(alpha=0.3)
axes[1].plot(hist_full['train_acc'], '-o', label='Train'); axes[1].plot(hist_full['val_acc'], '-s', label='Val')
axes[1].set_title('Accuracy', fontweight='bold'); axes[1].set_xlabel('Epoch'); axes[1].legend(); axes[1].grid(alpha=0.3)
axes[2].plot(hist_full['lr'], '-o', color='purple')
axes[2].set_title('LR schedule', fontweight='bold'); axes[2].set_xlabel('Epoch'); axes[2].grid(alpha=0.3)
plt.suptitle('Tuned CNN — full 25-epoch retraining', fontweight='bold')
plt.tight_layout()
plt.savefig('hp_full_retrain_curves.png', dpi=150, bbox_inches='tight')
plt.show()

## 9. Test-set evaluation (first and only touch)

In [ ]:
test_loader = DataLoader(image_datasets['test'], batch_size=best_config['batch_size'],
                         shuffle=False, num_workers=0)

tuned_model = TunableImprovedCNN(num_classes=num_classes, dropout_rate=best_config['dropout_rate']).to(device)
tuned_model.load_state_dict(full_result['best_weights'])
tuned_model.eval()

y_true, y_pred, y_prob = [], [], []
with torch.no_grad():
    for inputs, labels in test_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        outputs = tuned_model(inputs)
        probs = F.softmax(outputs, dim=1)
        _, preds = torch.max(outputs, 1)
        y_true.extend(labels.cpu().numpy())
        y_pred.extend(preds.cpu().numpy())
        y_prob.extend(probs.cpu().numpy())
y_true = np.array(y_true); y_pred = np.array(y_pred); y_prob = np.array(y_prob)

tuned_test_acc = (y_true == y_pred).mean()
print(f'Test accuracy: {tuned_test_acc:.4f}')
print('\nClassification report:')
print(classification_report(y_true, y_pred, target_names=class_names, digits=4))

In [ ]:
cm     = confusion_matrix(y_true, y_pred)
cm_pct = cm.astype(float) / cm.sum(axis=1, keepdims=True)
fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))
sns.heatmap(cm,     annot=True, fmt='d',   cmap='Purples',
            xticklabels=class_names, yticklabels=class_names, ax=axes[0])
axes[0].set_title('Confusion matrix — counts', fontweight='bold')
axes[0].set_xlabel('Predicted'); axes[0].set_ylabel('True')
sns.heatmap(cm_pct, annot=True, fmt='.2f', cmap='Purples',
            xticklabels=class_names, yticklabels=class_names, ax=axes[1], vmin=0, vmax=1)
axes[1].set_title('Confusion matrix — row-normalised', fontweight='bold')
axes[1].set_xlabel('Predicted'); axes[1].set_ylabel('True')
plt.tight_layout()
plt.savefig('hp_tuned_confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
p, r, f1, sup = precision_recall_fscore_support(y_true, y_pred, labels=range(num_classes), zero_division=0)
metrics_df = pd.DataFrame({'class': class_names, 'precision': p, 'recall': r, 'f1': f1, 'support': sup})
print(metrics_df.round(4).to_string(index=False))

fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(num_classes); w = 0.27
ax.bar(x - w, metrics_df['precision'], w, label='Precision', color='#1f77b4')
ax.bar(x,     metrics_df['recall'],    w, label='Recall',    color='#ff7f0e')
ax.bar(x + w, metrics_df['f1'],        w, label='F1',        color='#2ca02c')
ax.set_xticks(x); ax.set_xticklabels(class_names); ax.set_ylim(0, 1.05)
ax.set_title('Per-class precision / recall / F1 — Tuned CNN', fontweight='bold')
ax.set_ylabel('Score'); ax.legend(); ax.grid(axis='y', alpha=0.3)
for i in range(num_classes):
    ax.text(i - w, metrics_df['precision'][i] + 0.01, f'{metrics_df["precision"][i]:.2f}', ha='center', fontsize=8)
    ax.text(i,     metrics_df['recall'][i]    + 0.01, f'{metrics_df["recall"][i]:.2f}',    ha='center', fontsize=8)
    ax.text(i + w, metrics_df['f1'][i]        + 0.01, f'{metrics_df["f1"][i]:.2f}',        ha='center', fontsize=8)
plt.tight_layout()
plt.savefig('hp_tuned_per_class_metrics.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ROC curves
fig, ax = plt.subplots(figsize=(7.5, 6))
for i, cls in enumerate(class_names):
    y_bin = (y_true == i).astype(int)
    fpr, tpr, _ = roc_curve(y_bin, y_prob[:, i])
    ax.plot(fpr, tpr, linewidth=2, label=f'{cls} (AUC = {auc(fpr, tpr):.3f})')
ax.plot([0, 1], [0, 1], 'k--', alpha=0.4)
ax.set_xlabel('False positive rate'); ax.set_ylabel('True positive rate')
ax.set_title('One-vs-rest ROC curves — Tuned CNN', fontweight='bold')
ax.legend(loc='lower right'); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('hp_tuned_roc_curves.png', dpi=150, bbox_inches='tight')
plt.show()

## 10. Final three-way comparison — Baseline vs Improved vs Tuned

This is the headline table and chart for the report. Loads metric JSONs saved by the prior notebooks; if any file is missing, that row is skipped but the rest still render.

In [ ]:
p_macro, r_macro, f1_macro, _ = precision_recall_fscore_support(y_true, y_pred, average='macro', zero_division=0)
tuned_metrics = {
    'model_name'    : 'Tuned CNN (grid search)',
    'best_val_acc'  : float(full_result['best_val_acc']),
    'test_acc'      : float(tuned_test_acc),
    'test_precision_macro': float(p_macro),
    'test_recall_macro'   : float(r_macro),
    'test_f1_macro'       : float(f1_macro),
    'best_config'   : best_config,
    'class_names'   : class_names,
}
with open('tuned_metrics.json', 'w') as f:
    json.dump(tuned_metrics, f, indent=2)
print('Saved tuned_metrics.json')

In [ ]:
rows = []
for fname, label in [('baseline_metrics.json', 'Baseline CNN'),
                      ('improved_metrics.json', 'Improved CNN'),
                      ('tuned_metrics.json',    'Tuned CNN')]:
    if os.path.exists(fname):
        with open(fname) as f:
            m = json.load(f)
        rows.append({
            'Model'    : label,
            'Val Acc'  : m.get('best_val_acc'),
            'Test Acc' : m.get('test_acc'),
            'Test Prec (macro)' : m.get('test_precision_macro'),
            'Test Rec (macro)'  : m.get('test_recall_macro'),
            'Test F1 (macro)'   : m.get('test_f1_macro'),
        })
    else:
        print(f'(skipped: {fname} not found)')

comp_df = pd.DataFrame(rows)
print('\nFinal comparison:')
print(comp_df.round(4).to_string(index=False))
comp_df.to_csv('final_comparison.csv', index=False)

In [ ]:
if len(comp_df) >= 2:
    metrics_to_plot = ['Val Acc', 'Test Acc', 'Test Prec (macro)', 'Test Rec (macro)', 'Test F1 (macro)']
    fig, ax = plt.subplots(figsize=(13, 5.5))
    x = np.arange(len(metrics_to_plot)); w = 0.8 / len(comp_df)
    colors = ['#999999', '#2ca02c', '#9467bd']
    for i, (_, row) in enumerate(comp_df.iterrows()):
        vals = [row[m] for m in metrics_to_plot]
        bars = ax.bar(x + i*w - w*(len(comp_df)-1)/2, vals, w, label=row['Model'], color=colors[i % 3])
        for b, v in zip(bars, vals):
            ax.text(b.get_x() + b.get_width()/2, v + 0.005, f'{v:.3f}', ha='center', fontsize=8)
    ax.set_xticks(x); ax.set_xticklabels(metrics_to_plot)
    ax.set_ylabel('Score'); ax.set_ylim(0, 1.1)
    ax.set_title('Three-way model comparison on the held-out test set', fontweight='bold')
    ax.legend(); ax.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    plt.savefig('final_comparison_bars.png', dpi=150, bbox_inches='tight')
    plt.show()

In [ ]:
# Final printed summary — paste-friendly for the report
print('=' * 60)
print('FABRIC DEFECT CLASSIFICATION — FINAL SUMMARY')
print('=' * 60)
print('\nBest hyperparameters found:')
for k, v in best_config.items():
    print(f'  {k:<14s}: {v}')
print('\nTest-set performance:')
print(comp_df.round(4).to_string(index=False))
print('=' * 60)